In [9]:
# Cell 1: Fix — về root trước rồi clone
import os
os.chdir('/kaggle/working')  # ← Về đây trước!

!git clone -b model/sentiment-training-setup \
    https://github.com/nhienthai/AI_in_DevOps-DataOps-MLOps_Final_Project.git

os.chdir('/kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project')
print("Current dir:", os.getcwd())

!pip install -q -r requirements.txt


Cloning into 'AI_in_DevOps-DataOps-MLOps_Final_Project'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (376/376), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 376 (delta 147), reused 351 (delta 122), pack-reused 0 (from 0)
Receiving objects: 100% (376/376), 170.63 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (147/147), done.
Current dir: /kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project


In [10]:
# Cell 2: Train XLM-RoBERTa (10 epochs + early stopping)
!python scripts/train_model.py \
    --model-type transformer \
    --model-name xlm-roberta-base \
    --dataset tridm/UIT-VSFC \
    --epochs 10 \
    --batch-size 16 \
    --lr 2e-5 \
    --output-dir ./artifacts/xlm-roberta \
    --mlflow-uri sqlite:///mlflow.db


2026/08/12 16:11:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/12 16:11:08 INFO mlflow.store.db.utils: Updating database tables
2026/08/12 16:11:10 INFO mlflow.tracking.fluent: Experiment with name 'sentiment-analysis-uit-vsfc' does not exist. Creating a new experiment.
=== Starting Training (TRANSFORMER) ===
Dataset: tridm/UIT-VSFC
Model Name: xlm-roberta-base
MLflow URI: sqlite:///mlflow.db
INFO:src.sentiment.training.train:Loading dataset 'tridm/UIT-VSFC'...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tridm/UIT-VSFC/resolve/main/README.md "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/tridm/UIT-VSFC "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tridm/UIT-VSFC/resolve/3889e3161ad12a9433d9e5b8774dd477e137c082/UIT-VSFC.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/tridm/UIT-VSFC/

In [12]:
!python scripts/evaluate_model.py \
    --model-path ./artifacts/xlm-roberta/xlm-roberta \
    --split test \
    --show-wrong 20 \
    --output-csv ./artifacts/eval_results.csv


🖥️  Device: CUDA

📂 Loading 'tridm/UIT-VSFC' [test]...
   Samples: 3166
   NEGATIVE: 1409 (44.5%)
   NEUTRAL: 167 (5.3%)
   POSITIVE: 1590 (50.2%)

📦 Loading model from: ./artifacts/xlm-roberta/xlm-roberta
Loading weights: 100%|█| 201/201 [00:00<00:00, 1530.21it/s, Materializing param=

🔮 Running inference on 3166 samples...

📊 Confusion Matrix (rows=Actual, cols=Predicted):
            NEGATIVE   NEUTRAL  POSITIVE
----------------------------------------
NEGATIVE        1365        17        27
NEUTRAL           32        89        46
POSITIVE          55        26      1509

📈 Classification Report:
              precision    recall  f1-score   support

    NEGATIVE     0.9401    0.9688    0.9542      1409
     NEUTRAL     0.6742    0.5329    0.5953       167
    POSITIVE     0.9539    0.9491    0.9515      1590

    accuracy                         0.9359      3166
   macro avg     0.8561    0.8169    0.8337      3166
weighted avg     0.9330    0.9359    0.9339      3166


❌ Misclas

In [13]:
# Cell nén Model Weights và MLflow Database
!zip -r model_weights.zip ./artifacts/xlm-roberta/xlm-roberta
!zip -r mlflow_db.zip mlflow.db ./artifacts/eval_results.csv


  adding: artifacts/xlm-roberta/xlm-roberta/ (stored 0%)
  adding: artifacts/xlm-roberta/xlm-roberta/model.safetensors (deflated 26%)
  adding: artifacts/xlm-roberta/xlm-roberta/tokenizer_config.json (deflated 47%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/ (stored 0%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/model.safetensors (deflated 26%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/training_args.bin (deflated 53%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/scheduler.pt (deflated 61%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/config.json (deflated 53%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/optimizer.pt (deflated 71%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/rng_state.pth (deflated 26%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/trainer_state.json (deflated 74%)
  adding: artifacts/xlm-roberta/xlm-roberta/checkpoint-3580/scaler.pt (deflated 

In [15]:
import os
from IPython.display import FileLink, display

# Về thư mục làm việc gốc của Kaggle
os.chdir('/kaggle/working')

# Chuyển file zip ra thư mục gốc Kaggle
!mv /kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project/model_weights.zip /kaggle/working/ 2>/dev/null || true
!mv /kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project/mlflow_db.zip /kaggle/working/ 2>/dev/null || true

# Hiển thị link tải
display(FileLink('model_weights.zip'))
display(FileLink('mlflow_db.zip'))


/kaggle/working/model_weights.zip

/kaggle/working/mlflow_db.zip